In [3]:
from torchvision import transforms

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = transforms.Compose([
    # 1) Framing & jarak kamera
    transforms.RandomResizedCrop(224, scale=(0.80, 1.00), ratio=(0.90, 1.10)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),

    # 2) Kondisi lapangan: lighting & WB kamera (dibuat moderat)
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03)
    ], p=0.80),

    # 3) Lighting ekstrem kadang terjadi (ringan aja)
    transforms.RandomAutocontrast(p=0.10),

    # 4) Kamera blur sedikit (opsional tapi berguna)
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.12),

    # 5) ke tensor & normalize
    transforms.ToTensor(),
    transforms.Normalize(mean, std),

    # 6) Occlusion / background ganggu (daun lain di belakang)
    transforms.RandomErasing(
        p=0.25, scale=(0.02, 0.08), ratio=(0.3, 3.3), value='random'
    ),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


In [1]:
import sys, torch
print("Python:", sys.executable)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


Python: e:\semester7\comvis\deteksidaun\venv\Scripts\python.exe
CUDA: True
GPU: NVIDIA GeForce GTX 1650


In [8]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.datasets import ImageFolder
import timm
from tqdm import tqdm

# =========================
# CONFIG
# =========================
PROJECT_ROOT = r"E:\semester7\comvis\deteksidaun"
TRAIN_DIR = os.path.join(PROJECT_ROOT, "dataset/split/train")
VAL_DIR   = os.path.join(PROJECT_ROOT, "dataset/split/val")

OUT_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_OUT   = os.path.join(OUT_DIR, "best_model.pth")
CLASSES_OUT = os.path.join(OUT_DIR, "classes.pt")

BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-4              # ✅ lebih aman dari 3e-4 (kalau sudah stabil boleh balik 3e-4)
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.05
DROP_RATE = 0.2
PATIENCE = 6
NUM_WORKERS = 0

GRAD_CLIP_NORM = 1.0   # ✅ cegah grad meledak (NaN)

# =========================
# DEVICE
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# DATASET (pakai train_tf & val_tf yang sudah kamu buat)
# =========================
if "train_tf" not in globals() or "val_tf" not in globals():
    raise RuntimeError("train_tf / val_tf belum didefinisikan. Jalankan cell augmentasi dulu ya.")

if not os.path.isdir(TRAIN_DIR):
    raise FileNotFoundError(f"TRAIN_DIR tidak ditemukan: {TRAIN_DIR}")
if not os.path.isdir(VAL_DIR):
    raise FileNotFoundError(f"VAL_DIR tidak ditemukan: {VAL_DIR}")

train_ds = ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds   = ImageFolder(VAL_DIR, transform=val_tf)

torch.save(train_ds.classes, CLASSES_OUT)
num_classes = len(train_ds.classes)
print("Classes:", train_ds.classes)

# =========================
# WEIGHTED SAMPLER (imbalance)
# =========================
targets = np.array(train_ds.targets)
class_counts = np.bincount(targets)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[targets]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

print("Class counts:", dict(zip(train_ds.classes, class_counts.tolist())))

# =========================
# DATALOADER
# =========================
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# =========================
# MODEL
# =========================
model = timm.create_model(
    "convnext_small",
    pretrained=True,
    num_classes=num_classes,
    drop_rate=DROP_RATE
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

# ✅ MATIKAN AMP supaya gak NaN
use_amp = False

# =========================
# EVALUATE
# =========================
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

# =========================
# TRAIN + EARLY STOPPING
# =========================
best_val_acc = 0.0
wait = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        # ✅ guard NaN/inf
        if not torch.isfinite(loss):
            print("⚠️ Loss NaN/Inf terdeteksi! Stop training.")
            print("Loss:", loss.item())
            raise RuntimeError("NaN/Inf loss")

        loss.backward()

        # ✅ cegah grad meledak
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        optimizer.step()

        train_loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        train_correct += (pred == y).sum().item()
        train_total += y.size(0)

        pbar.set_postfix(loss=float(loss.item()))

    train_loss = train_loss_sum / train_total
    train_acc  = train_correct / train_total

    val_loss, val_acc = evaluate(val_loader)
    scheduler.step(val_loss)
    lr_now = optimizer.param_groups[0]["lr"]

    print(
        f"[Epoch {epoch:02d}] "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={lr_now:.2e}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_OUT)
        print(f"✅ Saved BEST -> {MODEL_OUT} (best_val_acc={best_val_acc:.4f})")
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"⛔ Early stopping di epoch {epoch}. Best val_acc={best_val_acc:.4f}")
            break

print("\n✅ Training selesai")
print("Best val_acc:", best_val_acc)
print("Model saved:", MODEL_OUT)
print("Classes saved:", CLASSES_OUT)


Device: cuda
Classes: ['Bercak Daun', 'Daun Sehat', 'Hawar Daun', 'Karat Daun']
Class counts: {'Bercak Daun': 401, 'Daun Sehat': 813, 'Hawar Daun': 802, 'Karat Daun': 914}


Epoch 1/50: 100%|██████████| 184/184 [23:37<00:00,  7.70s/it, loss=0.365]


[Epoch 01] train_loss=0.5367 train_acc=0.8573 | val_loss=0.3765 val_acc=0.9089 | lr=1.00e-04
✅ Saved BEST -> E:\semester7\comvis\deteksidaun\models\best_model.pth (best_val_acc=0.9089)


Epoch 2/50: 100%|██████████| 184/184 [23:51<00:00,  7.78s/it, loss=0.206]


[Epoch 02] train_loss=0.3888 train_acc=0.9242 | val_loss=0.3357 val_acc=0.9329 | lr=1.00e-04
✅ Saved BEST -> E:\semester7\comvis\deteksidaun\models\best_model.pth (best_val_acc=0.9329)


Epoch 3/50: 100%|██████████| 184/184 [23:42<00:00,  7.73s/it, loss=0.219]


... (31 outputs left out)

Epoch 19/50: 100%|██████████| 184/184 [23:32<00:00,  7.68s/it, loss=0.201]


[Epoch 19] train_loss=0.2076 train_acc=0.9973 | val_loss=0.3166 val_acc=0.9696 | lr=1.25e-05


Epoch 20/50: 100%|██████████| 184/184 [23:32<00:00,  7.68s/it, loss=0.201]


[Epoch 20] train_loss=0.2074 train_acc=0.9980 | val_loss=0.2976 val_acc=0.9712 | lr=6.25e-06
⛔ Early stopping di epoch 20. Best val_acc=0.9728

✅ Training selesai
Best val_acc: 0.9728434504792333
Model saved: E:\semester7\comvis\deteksidaun\models\best_model.pth
Classes saved: E:\semester7\comvis\deteksidaun\models\classes.pt


In [ ]:
# ── Distribusi ASLI dataset ──────────────────────────────────
print("=== Distribusi ASLI train dataset ===")
targets = np.array(train_ds.targets)
class_counts = np.bincount(targets)
total_ori = len(targets)
for i, (cls, cnt) in enumerate(zip(train_ds.classes, class_counts)):
    print(f"  {cls}: {cnt} ({cnt/total_ori*100:.1f}%)")

print()

# ── Distribusi SETELAH sampler (simulasi 1 epoch penuh) ──────
print("=== Distribusi SETELAH WeightedRandomSampler (1 epoch) ===")
sampled_labels = Counter()
for _, y in train_loader:
    for label in y.tolist():
        sampled_labels[label] += 1

total_sampled = sum(sampled_labels.values())
for label_idx, count in sorted(sampled_labels.items()):
    cls = train_ds.classes[label_idx]
    print(f"  {cls}: {count} ({count/total_sampled*100:.1f}%)")